In [16]:
import os
import cv2
import math
import random
import numpy as np
import tensorflow as tf
import keras
from collections import deque
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras.layers import (
    Input, Dense, Dropout, Flatten, LSTM, Bidirectional,
    TimeDistributed, BatchNormalization, GlobalAveragePooling2D,
    MultiHeadAttention, LayerNormalization, Reshape, Permute,
    Lambda, Add
)
from keras.models import Sequential, Model
from keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.applications.mobilenet_v2 import MobileNetV2


In [17]:
# =============================================================================
# 1. HYPERPARAMETERS
# =============================================================================

# ▶ 64→112: Tăng độ phân giải để model nhận diện tốt hơn trên CCTV (thường
#   có nhiễu, góc rộng) và RWF-2000 (quay bằng camera cầm tay, nhiều chuyển
#   động). 112 là bội số của stride MobileNetV2, tránh mất thông tin biên.
IMAGE_HEIGHT, IMAGE_WIDTH = 112, 112

# ▶ 16→20: Tăng cửa sổ thời gian để bắt được các pha hành động dài hơn.
#   Hockey Fight thường có chuỗi đánh dài ~1-2s (≈30 frame). Với 20 frame
#   trích đều, model thấy nhiều context temporal hơn mà vẫn tiết kiệm RAM.
SEQUENCE_LENGTH = 20

# Classes — mở rộng khi finetune multi-dataset
CLASSES_LIST = ["NonViolence", "Violence"]

# ▶ 8→16: Batch lớn hơn ổn định gradient hơn cho dataset nhỏ (Hockey Fight
#   ~1000 clip). Nếu OOM trên GPU <8GB, giảm về 8.
BATCH_SIZE = 16

EPOCHS = 60

DATASET_DIR = "/kaggle/input/datasets/mohamedmustafa/real-life-violence-situations-dataset/Real Life Violence Dataset"   # Thay bằng đường dẫn thực tế



In [18]:
# =============================================================================
# 2. FRAME EXTRACTION  (cải tiến: thêm random crop + color jitter khi train)
# =============================================================================

def frames_extraction(video_path, augment: bool = False):
    """
    Trích SEQUENCE_LENGTH frame phân bố đều từ video.

    augment=True: áp dụng horizontal flip + slight brightness jitter
    (dùng khi train để giúp model robust với CCTV góc khác nhau).
    """
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

    if video_frames_count == 0:
        video_reader.release()
        return frames_list

    skip_frames_window = max(int(video_frames_count / SEQUENCE_LENGTH), 1)

    # Random horizontal flip quyết định per-video (nhất quán cả sequence)
    do_flip = augment and random.random() > 0.5
    # Brightness factor: ±20%
    brightness = 1.0 + random.uniform(-0.2, 0.2) if augment else 1.0

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break

        # ▶ Resize lên 112×112 (thay vì 64×64)
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        resized_frame = resized_frame.astype(np.float32)

        if do_flip:
            resized_frame = resized_frame[:, ::-1, :]
        if augment:
            resized_frame = np.clip(resized_frame * brightness, 0, 255)

        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list

In [19]:
# =============================================================================
# 3. DATASET CREATION
# =============================================================================

def create_dataset(augment_train: bool = True):
    features, labels, video_files_paths = [], [], []

    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f"Extracting Data of Class: {class_name}")
        class_dir = os.path.join(DATASET_DIR, class_name)
        files_list = os.listdir(class_dir)

        for file_name in files_list:
            video_file_path = os.path.join(class_dir, file_name)
            frames = frames_extraction(video_file_path, augment=False)
            if len(frames) == SEQUENCE_LENGTH:
                features.append(frames)
                labels.append(class_index)
                video_files_paths.append(video_file_path)

    features = np.asarray(features, dtype=np.float32)
    labels = np.array(labels)
    return features, labels, video_files_paths



In [20]:
# =============================================================================
# 4. BACKBONE — MobileNetV2 với chiến lược fine-tune tốt hơn
# =============================================================================

def build_mobilenet_backbone():
    """
    ▶ Unfreeze [-60:] thay vì [-40:]:
      MobileNetV2 có 154 layer. Với dataset mới (CCTV/RWF-2000), các block
      InvertedResidual cuối (block 14-16) cần học lại pattern motion blur,
      grain camera khác với ImageNet. Mở thêm 20 layer giúp adapt tốt hơn
      mà vẫn giữ low-level feature frozen (tiết kiệm compute + tránh
      catastrophic forgetting).

    ▶ include_top=False + GlobalAveragePooling trong TimeDistributed:
      Output shape (None, 3, 3, 1280) → GAP → (None, 1280) per frame
      Thay Flatten (None, 11520) vì GAP bất biến với spatial shift — quan
      trọng với CCTV có camera rung.
    """
    mobilenet = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3)
    )

    mobilenet.trainable = True
    # Freeze tất cả trừ 60 layer cuối
    for layer in mobilenet.layers[:-60]:
        layer.trainable = False

    # Thêm GAP trực tiếp vào backbone để TimeDistributed xử lý gọn
    x = mobilenet.output
    x = GlobalAveragePooling2D()(x)          # (batch, 1280)
    backbone = Model(mobilenet.input, x, name="mobilenet_gap")
    return backbone

In [21]:
# =============================================================================
# 5. TEMPORAL ATTENTION LAYER
# =============================================================================

class TemporalAttention(keras.layers.Layer):
    """
    ▶ Lý do thêm attention:
      Không phải frame nào trong sequence cũng chứa violence. Frame đầu/cuối
      thường là transition. Attention học weigh cao hơn cho frame có motion
      mạnh (punch, kick). Giúp mô hình robust hơn trên RWF-2000 (quay tay,
      nhiều camera shake ở frame không liên quan).

    Cơ chế: Scaled dot-product self-attention trên trục temporal.
    Input: (batch, SEQUENCE_LENGTH, lstm_dim)
    Output: (batch, SEQUENCE_LENGTH, lstm_dim)  — weighted by attention score
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W = Dense(units, use_bias=False)
        self.V = Dense(1)

    def call(self, hidden_states):
        # hidden_states: (batch, T, D)
        score = tf.nn.tanh(self.W(hidden_states))   # (batch, T, units)
        attention_weights = tf.nn.softmax(self.V(score), axis=1)  # (batch, T, 1)
        context = attention_weights * hidden_states  # (batch, T, D)
        return context  # vẫn giữ shape để dùng với return_sequences=True

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


In [22]:
# =============================================================================
# 6. BUILD MODEL
# =============================================================================

def create_model():
    """
    Kiến trúc tổng thể:
    Input (B, T, H, W, 3)
        → TimeDistributed(MobileNetV2+GAP)  → (B, T, 1280)
        → BatchNorm + Dropout(0.5)
        → BiLSTM(128) return_sequences=True  → (B, T, 256)
        → TemporalAttention
        → BatchNorm + Dropout(0.5)
        → BiLSTM(64) return_sequences=False  → (B, 128)
        → BatchNorm + Dropout(0.4)
        → Dense(256, relu, L2)
        → BatchNorm + Dropout(0.4)
        → Dense(128, relu, L2)
        → BatchNorm
        → Dense(num_classes, softmax)
    """
    backbone = build_mobilenet_backbone()

    sequence_input = Input(shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3),
                           name="sequence_input")

    # --- Feature Extraction ---
    # ▶ GlobalAveragePooling2D (trong backbone) thay Flatten:
    #   Flatten(3×3×1280)=11520 dim → GAP=1280 dim, giảm 9× params,
    #   tránh overfit trên dataset nhỏ như Hockey Fight (~900 clip train).
    x = TimeDistributed(backbone, name="td_mobilenet")(sequence_input)
    x = TimeDistributed(BatchNormalization(), name="td_bn")(x)

    # ▶ Dropout 0.25→0.5 trước LSTM:
    #   Dataset mới nhỏ hơn Real-Life Violence (1000 clip vs 2000).
    #   Dropout cao hơn cần thiết để tránh overfit backbone features.
    x = Dropout(0.5, name="dropout_features")(x)

    # --- Temporal Modeling: 2-layer BiLSTM ---
    # ▶ 32→128 units: Tăng capacity để học temporal pattern phức tạp hơn.
    #   RWF-2000 có violence đa dạng (đánh nhau, đám đông, vũ khí). 32 unit
    #   quá nhỏ để encode đủ context. 128 đã đủ mà không quá nặng.
    #
    # ▶ 1→2 BiLSTM layers: Layer 1 học low-level motion patterns;
    #   Layer 2 học high-level event structure (buildup → peak → aftermath).
    #   Thường thấy trong SOTA action recognition (PoseConv3D, SlowFast).
    x = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
        name="bilstm_1"
    )(x)
    x = BatchNormalization(name="bn_lstm1")(x)

    # ▶ Temporal Attention sau LSTM layer 1:
    #   Cho phép model tập trung vào frame quyết định (ví dụ: moment impact).
    x = TemporalAttention(units=256, name="temporal_attention")(x)

    x = Bidirectional(
        LSTM(64, return_sequences=False, dropout=0.3, recurrent_dropout=0.2),
        name="bilstm_2"
    )(x)
    x = BatchNormalization(name="bn_lstm2")(x)
    x = Dropout(0.4, name="dropout_lstm")(x)

    # --- Classification Head ---
    # ▶ Đơn giản hóa từ 256→128→64→32 xuống 256→128:
    #   Nhiều Dense layer nhỏ liên tiếp ≈ một Dense layer lớn hơn nhưng
    #   thêm overhead gradient vanishing. Với representation quality tốt
    #   từ BiLSTM, head đơn giản hơn generalize tốt hơn.
    #
    # ▶ L2 regularization 1e-4:
    #   Cần thiết khi finetune trên dataset nhỏ (Hockey ~1000 clip).
    x = Dense(256, activation="relu",
               kernel_regularizer=l2(1e-4), name="dense_1")(x)
    x = BatchNormalization(name="bn_dense1")(x)
    x = Dropout(0.4, name="dropout_dense1")(x)

    x = Dense(128, activation="relu",
               kernel_regularizer=l2(1e-4), name="dense_2")(x)
    x = BatchNormalization(name="bn_dense2")(x)
    x = Dropout(0.4, name="dropout_dense2")(x)

    output = Dense(len(CLASSES_LIST), activation="softmax", name="output")(x)

    model = Model(inputs=sequence_input, outputs=output, name="MoBiLSTM_v2")
    model.summary()
    return model


In [23]:
# =============================================================================
# 7. COMPILE & CALLBACKS
# =============================================================================

def compile_model(model):
    """
    ▶ SGD → Adam (lr=1e-4, clipnorm=1.0):
      - Adam hội tụ nhanh hơn SGD khi finetune pretrained network.
      - clipnorm=1.0 ngăn gradient explode trong BiLSTM (thường xảy ra
        khi sequence dài hoặc recurrent_dropout cao).
      - lr=1e-4: Learning rate nhỏ để không phá vỡ MobileNet weights đã học.
        Nếu chỉ train head, có thể dùng 3e-4. Nếu unfreeze toàn bộ, dùng 5e-5.
    """
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    )
    model.compile(
        loss="categorical_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
        # ▶ Thêm AUC metric: quan trọng hơn accuracy khi dataset imbalanced
        #   (ví dụ CCTV surveillance có tỉ lệ violence thấp ~10-20%).
    )
    return model


def get_callbacks(model_save_path: str = "best_model.keras"):
    """
    ▶ Cosine Annealing thay ReduceLROnPlateau:
      Cosine decay giúp thoát local minima tốt hơn khi finetune. Với patience
      dài (15 epoch), RLRP có thể stuck. CosineDecay restart mỗi ~20 epoch.
    """
    cosine_lr = tf.keras.callbacks.LearningRateScheduler(
        lambda epoch: 1e-5 + 0.5 * (1e-4 - 1e-5) * (
            1 + math.cos(math.pi * (epoch % 20) / 20)
        ),
        verbose=0
    )

    # ▶ Patience 10→15: Dataset mới có thể cần nhiều epoch hơn để ổn định
    #   do domain shift (CCTV texture khác ImageNet).
    early_stopping = EarlyStopping(
        monitor="val_auc",          # Monitor AUC thay accuracy
        patience=15,
        restore_best_weights=True,
        mode="max"
    )

    # ▶ Lưu model tốt nhất theo val_auc
    checkpoint = ModelCheckpoint(
        filepath=model_save_path,
        monitor="val_auc",
        save_best_only=True,
        mode="max",
        verbose=1
    )

    return [early_stopping, checkpoint, cosine_lr]



In [24]:
# =============================================================================
# 8. TRAINING PIPELINE
# =============================================================================

def train(dataset_dir: str = "/kaggle/working/", model_save_path: str = "best_model.keras"):
    global DATASET_DIR
    DATASET_DIR = dataset_dir

    # Load & split data
    features, labels, _ = create_dataset()
    one_hot_labels = to_categorical(labels)

    features_train, features_test, labels_train, labels_test = train_test_split(
        features, one_hot_labels,
        test_size=0.15,       # ▶ 10→15%: Dataset nhỏ cần test set lớn hơn
        shuffle=True,
        stratify=labels,      # ▶ Thêm stratify: đảm bảo tỉ lệ class cân bằng
        random_state=42
    )

    print(f"Train: {features_train.shape}, Test: {features_test.shape}")

    # Build & compile
    model = create_model()
    model = compile_model(model)
    callbacks = get_callbacks(model_save_path)

    # Fit
    history = model.fit(
        x=features_train,
        y=labels_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        validation_split=0.15,   # ▶ Tách val từ train
        callbacks=callbacks
    )

    # Evaluate
    print("\n=== Test Set Evaluation ===")
    model.evaluate(features_test, labels_test)

    return model, history


In [25]:
# =============================================================================
# 9. INFERENCE (Realtime)
# =============================================================================

def predict_frames(video_file_path: str, output_file_path: str, model):
    """
    Sliding window inference với deque SEQUENCE_LENGTH.
    Confidence threshold: dùng max probability thay vì argmax trực tiếp
    để tránh false positive trên CCTV footage bình thường.
    """
    CONFIDENCE_THRESHOLD = 0.70  # ▶ Chỉ label Violence nếu prob > 70%

    video_reader = cv2.VideoCapture(video_file_path)
    original_video_width  = int(video_reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_video_height = int(video_reader.get(cv2.CAP_PROP_FRAME_HEIGHT))

    video_writer = cv2.VideoWriter(
        output_file_path,
        cv2.VideoWriter_fourcc('m', 'p', '4', 'v'),
        video_reader.get(cv2.CAP_PROP_FPS),
        (original_video_width, original_video_height)
    )

    frames_queue = deque(maxlen=SEQUENCE_LENGTH)
    predicted_class_name = ""
    confidence = 0.0

    while video_reader.isOpened():
        ok, frame = video_reader.read()
        if not ok:
            break

        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame.astype(np.float32) / 255.0
        frames_queue.append(normalized_frame)

        if len(frames_queue) == SEQUENCE_LENGTH:
            probs = model.predict(
                np.expand_dims(frames_queue, axis=0), verbose=0
            )[0]
            predicted_label = np.argmax(probs)
            confidence = probs[predicted_label]

            # ▶ Confidence threshold: giảm false positive trên CCTV
            if CLASSES_LIST[predicted_label] == "Violence" and confidence >= CONFIDENCE_THRESHOLD:
                predicted_class_name = f"Violence ({confidence:.0%})"
                color = (0, 0, 255)
            elif CLASSES_LIST[predicted_label] == "NonViolence":
                predicted_class_name = f"Normal ({confidence:.0%})"
                color = (0, 200, 0)
            else:
                predicted_class_name = f"Uncertain ({confidence:.0%})"
                color = (0, 165, 255)

        if predicted_class_name:
            cv2.putText(frame, predicted_class_name, (10, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 2, color, 4)

        video_writer.write(frame)

    video_reader.release()
    video_writer.release()
    print(f"Output saved to: {output_file_path}")


In [26]:
# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    model, history = train(
        dataset_dir="/kaggle/input/datasets/mohamedmustafa/real-life-violence-situations-dataset/Real Life Violence Dataset",
        model_save_path="violence_model_v2.keras"
    )

Extracting Data of Class: NonViolence


[h264 @ 0x6d6a45c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x6d6a45c0] error while decoding MB 98 31
[h264 @ 0x6d6a45c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x6d6a45c0] error while decoding MB 98 31
[h264 @ 0x6d6a45c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x6d6a45c0] error while decoding MB 98 31
[h264 @ 0x6d6a45c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x6d6a45c0] error while decoding MB 98 31


Extracting Data of Class: Violence
Train: (1700, 20, 112, 112, 3), Test: (300, 20, 112, 112, 3)


/tmp/ipykernel_57/3140031303.py:19: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobilenet = MobileNetV2(
I0000 00:00:1778214257.604257      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778214257.610673      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


...

 Trainable params: 3,698,947 (14.11 MB)

 Non-trainable params: 306,368 (1.17 MB)

Epoch 1/60


I0000 00:00:1778214429.677098   14822 cuda_dnn.cc:529] Loaded cuDNN version 91002


91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.5231 - auc: 0.5201 - loss: 1.2700
Epoch 1: val_auc improved from -inf to 0.67328, saving model to violence_model_v2.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 258s 1s/step - accuracy: 0.5233 - auc: 0.5205 - loss: 1.2689 - val_accuracy: 0.5569 - val_auc: 0.6733 - val_loss: 0.7163 - learning_rate: 1.0000e-04
Epoch 2/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 650ms/step - accuracy: 0.5435 - auc: 0.5766 - loss: 1.0909
Epoch 2: val_auc improved from 0.67328 to 0.73501, saving model to violence_model_v2.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 64s 700ms/step - accuracy: 0.5438 - auc: 0.5770 - loss: 1.0903 - val_accuracy: 0.5569 - val_auc: 0.7350 - val_loss: 0.7205 - learning_rate: 9.9446e-05
Epoch 3/60
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 656ms/step - accuracy: 0.6371 - auc: 0.6890 - loss: 0.8620
Epoch 3: val_auc improved from 0.73501 to 0.76601, saving model to violence_model_v2.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 64s 707ms/step - accuracy: 0.6373 - auc: 0.6893 - loss: 0.86

In [ ]:
# =============================================================================
# 10. LEARNING CURVE PLOT
# =============================================================================

if "history" not in globals():
    raise RuntimeError("Chạy lại cell train() trước khi vẽ learning curve.")

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train_loss", marker="o")
plt.plot(history.history["val_loss"], label="val_loss", marker="o")
plt.title("Learning Curve - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train_accuracy", marker="o")
plt.plot(history.history["val_accuracy"], label="val_accuracy", marker="o")
plt.title("Learning Curve - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()
</VSCode.Cell>
<VSCode.Cell language="python">
# =============================================================================
# 11. CONFUSION MATRIX ON TEST SET
# =============================================================================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Tạo lại test set và tính ma trận nhầm lẫn...")
features, labels, _ = create_dataset(augment_train=False)
_, features_test, _, labels_test = train_test_split(
    features, labels,
    test_size=0.15,
    shuffle=True,
    stratify=labels,
    random_state=42
)

predictions = model.predict(features_test, verbose=0)
y_pred = np.argmax(predictions, axis=1)

cm = confusion_matrix(labels_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES_LIST)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix trên Test Set")
plt.tight_layout()
plt.show()

print("\nClassification report:")
print(classification_report(labels_test, y_pred, target_names=CLASSES_LIST, digits=4))
